In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd
import time

# ── Parámetros del problema (idénticos a MELIN) ──────────────
alpha  = 0.2        # Constante de difusión térmica [m²/s^(1/2)]
L      = 1.0        # Ancho de la pared [m]
N      = 10         # Número de divisiones espaciales
dx     = L / N      # Paso espacial
T0     = 100.0      # Temperatura inicial [°C]

# ── Parámetros temporales ────────────────────────────────────
dt     = 0.05       # Paso de tiempo [s]  (mismo que MELIN h=0.05)
t_end  = 30.0       # Tiempo final [s]
NPT    = int(t_end / dt)   # Número de pasos de tiempo

# ── Número de Fourier (parámetro de estabilidad) ─────────────
r = alpha**2 * dt / dx**2

# ── Grid espacial ────────────────────────────────────────────
x = np.linspace(0, L, N+1)   # N+1 puntos: x_0, x_1, ..., x_N

print(f"alpha  = {alpha}")
print(f"l      = {L} m")
print(f"N      = {N}  nodos")
print(f"dx     = {dx}")
print(f"dt     = {dt}")
print(f"NPT    = {NPT} pasos de tiempo")
print(f"r      = alpha^2*dt/dx^2 = {r:.4f}")
print()
if r <= 0.5:
    print(f"Criterio de estabilidad: r = {r:.4f} <= 0.5 -> ESTABLE")
else:
    print(f"Criterio de estabilidad: r = {r:.4f} > 0.5  -> INESTABLE")

alpha  = 0.2
l      = 1.0 m
N      = 10  nodos
dx     = 0.1
dt     = 0.05
NPT    = 600 pasos de tiempo
r      = alpha^2*dt/dx^2 = 0.2000

Criterio de estabilidad: r = 0.2000 <= 0.5 -> ESTABLE


In [2]:
def construir_matriz_explicita(N, r):
    """
    Construye la matriz explícita B del esquema de diferencias
    finitas explícito para la ecuación de difusión 1D.

    El paso temporal es simplemente:  T^{n+1} = B * T^n

    Frontera x=0  : Dirichlet  T=0  -> fila 0 de ceros (RHS=0)
    Nodos int.    : i=1,...,N-1 -> esquema explícito estandar
                    B[i,i-1]=r, B[i,i]=1-2r, B[i,i+1]=r
    Frontera x=l  : Neumann dT/dx=0 usando T_{N+1}=(4T_N-T_{N-1})/3
                    B[N,N-1]=r/3, B[N,N]=1-2r/3
    """
    size = N + 1   # indices 0..N

    B = np.zeros((size, size))

    # ── Fila 0: Dirichlet T(0,t) = 0 ────────────────────────
    B[0, 0] = 0.0   # T_0^{n+1} = 0 siempre

    # ── Filas interiores i = 1 ... N-1 ──────────────────────
    for i in range(1, N):
        B[i, i-1] =  r
        B[i, i  ] =  1.0 - 2.0*r
        B[i, i+1] =  r

    # ── Fila N: Neumann (dT/dx = 0 en x = l) ────────────────
    # Sustituyendo T_{N+1} = (4*T_N - T_{N-1})/3 en el esquema:
    #   T_N^{n+1} = r*T_{N-1}^n + (1-2r)*T_N^n + r*T_{N+1}^n
    #             = r*T_{N-1}^n + (1-2r)*T_N^n + r*(4T_N-T_{N-1})/3
    #             = (r - r/3)*T_{N-1}^n + (1-2r + 4r/3)*T_N^n
    #             = (2r/3)*T_{N-1}^n    + (1 - 2r/3)*T_N^n
    B[N, N-1] =  2.0*r / 3.0
    B[N, N  ] =  1.0 - 2.0*r / 3.0

    return B


B = construir_matriz_explicita(N, r)

print("Matriz B (explicita) — primeras 5x5:")
print(np.round(B[:5, :5], 4))


Matriz B (explicita) — primeras 5x5:
[[0.  0.  0.  0.  0. ]
 [0.2 0.6 0.2 0.  0. ]
 [0.  0.2 0.6 0.2 0. ]
 [0.  0.  0.2 0.6 0.2]
 [0.  0.  0.  0.2 0.6]]


In [3]:
# ── Condición inicial ────────────────────────────────────────
T_exp       = np.full(N+1, T0)   # T_i(t=0) = 100 para i=1..N
T_exp[0]    = 0.0                 # Dirichlet T(0,0) = 0

# ── Almacenamiento de la solución ───────────────────────────
t_vals      = np.arange(0, t_end + dt, dt)   # tiempos de salida
T_hist      = np.zeros((N+1, len(t_vals)))    # T[nodo, tiempo]
T_hist[:, 0] = T_exp.copy()

# ── Cronómetro ───────────────────────────────────────────────
t_inicio = time.perf_counter()

# ── Bucle temporal — un simple producto matriz-vector ────────
# T^{n+1} = B * T^n  (no hay sistema lineal que resolver)
T_actual = T_exp.copy()
for k in range(1, len(t_vals)):
    T_nuevo       = B @ T_actual       # avance explicito
    T_nuevo[0]    = 0.0               # reforzar Dirichlet
    T_hist[:, k]  = T_nuevo
    T_actual      = T_nuevo

t_fin    = time.perf_counter()
t_ejec   = t_fin - t_inicio

# ── Temperatura en la frontera aislada (x=l) ────────────────
T_frontera = (4*T_hist[N, :] - T_hist[N-1, :]) / 3.0

print(f"Integracion completada: {len(t_vals)} pasos")
print(f"=========================================")
print(f"  Tiempo de ejecucion : {t_ejec:.6f} segundos")
print(f"  Archivo generado    : EXP.dat")
print(f"=========================================")

Integracion completada: 601 pasos
  Tiempo de ejecucion : 0.000833 segundos
  Archivo generado    : EXP.dat


In [4]:
# ── EXP.dat: columnas = [t, T_0..T_{N-1}, T_frontera] ────────
# Misma convencion que CNPY.dat:
# T_hist[:N, :].T  ->  nodos Python 0..N-1 (incluye T0=0)
datos_exp = np.column_stack([
    t_vals,
    T_hist[:N, :].T,   # nodos 0..N-1 (T0=0 Dirichlet incluido)
    T_frontera         # temperatura virtual en x=l
])

# Encabezado con nombres de columnas
header = "t" + "".join([f"      T{i+1}" for i in range(N)]) + "  T_frontera"
np.savetxt('FTCSPY.dat', datos_exp, fmt='%10.4f', header=header, comments='')

print("Primeras 5 filas de FTCSPY.dat:")
df_preview = pd.DataFrame(
    datos_exp[:5],
    columns=["t"] + [f"T{i+1}" for i in range(N)] + ["T_frontera"]
)
display(df_preview.round(4))

Primeras 5 filas de FTCSPY.dat:


,t,T1,T2,T3,T4,T5,T6,T7,T8,T9,T10,T_frontera
0,0.00,0.0,100.00,100.00,100.00,100.00,100.0,100.0,100.0,100.0,100.0,100.0
1,0.05,0.0,80.00,100.00,100.00,100.00,100.0,100.0,100.0,100.0,100.0,100.0
2,0.10,0.0,68.00,96.00,100.00,100.00,100.0,100.0,100.0,100.0,100.0,100.0
3,0.15,0.0,60.00,91.20,99.20,100.00,100.0,100.0,100.0,100.0,100.0,100.0
4,0.20,0.0,54.24,86.56,97.76,99.84,100.0,100.0,100.0,100.0,100.0,100.0
